In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import gc

# -----------------------------------------
# 1. Sharpe Ratio Function (unchanged)
# -----------------------------------------
def calc_spread_return_sharpe(df: pd.DataFrame, portfolio_size: int = 200, toprank_weight_ratio: float = 2) -> float:
    def _calc_spread_return_per_day(g, portfolio_size, toprank_weight_ratio):
        assert g['Rank'].min() == 0
        assert g['Rank'].max() == len(g['Rank']) - 1
        n_stocks = len(g)
        ps = min(portfolio_size, n_stocks)
        top = g.sort_values(by='Rank')['Target'][:ps]
        bot = g.sort_values(by='Rank', ascending=False)['Target'][:ps]
        w = np.linspace(start=toprank_weight_ratio, stop=1, num=len(top))
        ret_long  = (top  * w).sum() / w.mean()
        ret_short = (bot  * w).sum() / w.mean()
        return ret_long - ret_short

    daily_spreads = df.groupby('Date').apply(_calc_spread_return_per_day,
                                             portfolio_size, toprank_weight_ratio)
    return daily_spreads.mean() / (daily_spreads.std() + 1e-8)


# -----------------------------------------
# 2. Load and preprocess augment.csv
# -----------------------------------------
data = pd.read_csv("augment_Date.csv")
data = (
    data
    .dropna()
    .sort_values(['Date', 'SecuritiesCode'])
    .reset_index(drop=True)
)

# -----------------------------------------
# 3. Manual train/test split (80/20 by rows)
# -----------------------------------------
# Sort data by pseudo-date (or real date if available)
data = data.sort_values(by="Date").reset_index(drop=True)

# Define split index
split_idx = int(len(data) * 0.8)

# Split without shuffling
train_df = data.iloc[:split_idx].reset_index(drop=True)
test_df  = data.iloc[split_idx:].reset_index(drop=True)

# (optional) re-sort each split for consistency
train_df = train_df.sort_values(['Date', 'SecuritiesCode']).reset_index(drop=True)
test_df  = test_df.sort_values(['Date', 'SecuritiesCode']).reset_index(drop=True)


# -----------------------------------------
# 4. Define features & target
# -----------------------------------------
# exclude any identifier cols
features = [c for c in data.columns if c not in ['Target', 'RowId', 'Date']]
X_train = train_df[features]
y_train = train_df['Target']

X_test  = test_df[features]
y_test  = test_df['Target']


# -----------------------------------------
# 5. Best hyperparameters (unchanged)
# -----------------------------------------
best_params={
    "colsample_bytree": 0.5,
    "learning_rate": 0.1,
    "max_depth": 15,
    "min_child_samples": 50,
    "n_estimators": 1000,
    "num_leaves": 255,
    "reg_alpha": 1.0,
    "reg_lambda": 0.1,
    "subsample": 1.0,
    "random_state": 42,
    "metric": "mae"
}

best_params_v2 = {
    "boosting_type": "gbdt",
    "num_leaves": 77,
    "max_depth": 19,
    "min_child_samples": 41,
    "learning_rate": 0.09953003012549799,
    "n_estimators": 2000,
    "colsample_bytree": 0.42197751965634284,
    "reg_alpha": 2.0880923278589756,
    "reg_lambda": 3.073547390938834,
    "subsample": 0.8832492105102278,
    "bagging_freq": 1,
    "random_state": 42,
    "metric": "mae"
}

# -----------------------------------------
# 6. Train on augment.csv train split
# -----------------------------------------
model = LGBMRegressor(**best_params_v2)
model.fit(X_train, y_train)
gc.collect()


# -----------------------------------------
# 7. Predict & evaluate
# -----------------------------------------
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
print(f"Test RMSE:  {rmse:.4f}")
print(f"Test MAE:   {mae:.4f}")

# prepare for sharpe
test_df = test_df.copy()
test_df['pred'] = y_pred
test_df['Rank'] = (
    test_df
    .groupby('Date')['pred']
    .rank(method='first', ascending=False) - 1
)

test_sharpe = calc_spread_return_sharpe(test_df)
print(f"Test Sharpe Ratio: {test_sharpe:.4f}")


# -----------------------------------------
# 8. (Optional) Save a “submission” for the hold-out set
# -----------------------------------------
submission = test_df[['Date', 'SecuritiesCode', 'Rank']]
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5457
[LightGBM] [Info] Number of data points in the train set: 625, number of used features: 28
[LightGBM] [Info] Start training from score 0.000540
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

C:\Users\zixuanmu\AppData\Local\Temp\1\ipykernel_31156\2042999114.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_spreads = df.groupby('Date').apply(_calc_spread_return_per_day,
